MOVE THIS TO THE END OF 01 FOR FUTURE EXPORTS

In [21]:
import pandas as pd, numpy as np, os

current_directory = os.getcwd()
Figures = os.path.join(current_directory, 'Manuscript Figures')
CSV = os.path.join(current_directory, 'CSV Files')
PKL = os.path.join(current_directory, 'PKL')

In [22]:
#select your tier - all figures for the manuscript use tier 4
tier = 4

print(f"Loading Tier {tier} data...")
try:
    df = pd.read_pickle(os.path.join(PKL, f'final_Tier{tier}.pkl'))
    print("File loaded!")
except FileNotFoundError:
    print(f"Error: final_Tier{tier}.pkl' not found.")

Loading Tier 4 data...
File loaded!


In [23]:
unique_bf = df['BF Value'].unique()
print(unique_bf)

[ 7.  5. nan  2.  1.  4.  8.  6.  0. 23.  9. 10. 45.  3. 56. 67. 12.]


In [24]:
def map_bf(value):
    """
    Map Beaufort values:
    - Keep valid values 0–12 as-is.
    - Map ranges like 23 → 2, 45 → 4, 56 → 5, 67 → 6.
    - Leave NaN as NaN.
    """
    if pd.isna(value):
        return np.nan
    try:
        v = int(value)
        if 0 <= v <= 12:
            return v
        else:
            return int(str(v)[0])  # take the first digit for borderline cases
    except (ValueError, TypeError):
        return np.nan

df['BF Value Mapped'] = df['BF Value'].apply(map_bf)
print(sorted(df['BF Value Mapped'].dropna().unique()))


[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 12.0]


In [25]:
df.columns

Index(['ID', 'LogBook ID', 'Page', 'Entry Date', 'Latitude',
       'Latitude_decimal', 'Longitude', 'Longitude_decimal', 'Depth',
       'Depth Unit', 'Bottom', 'Landmark', 'Ship Heading/Course',
       'Wind Direction', 'WD_Bearing', 'Wind Speed/Force', 'Sea State',
       'Cloud Cover', 'Weather', 'Ship Sightings',
       'Miscellaneous Observations', 'Entry Date Time', 'BF Value',
       'prev_time', 'prev_lat', 'prev_lon', 'delta_time_s', 'dlat_deg',
       'dlon_deg', 'coord_diff', 'Infilled', 'usable', 'gap_distance_km',
       'gap_days_missing', 'gap_type', 'infill_days_missing', 'infill_type',
       'infill_distance_km', 'Tier2_usable', 'Tier3_usable', 'Tier4_usable',
       'BF Value Mapped'],
      dtype='object')

In [26]:
df = df.drop(columns=['BF Value'])

df = df.rename(columns={'BF Value Mapped': 'BF Value'})

cols = list(df.columns)
insert_at = cols.index('Wind Speed/Force') + 1
cols.remove('BF Value')
cols.insert(insert_at, 'BF Value')
df = df[cols]

# 4. Save as PKL and CSV
out_base = "Final_tier4_2025-08-26"  
df.to_pickle(os.path.join(PKL, f"{out_base}.pkl"))
df.to_csv(os.path.join(CSV, f"{out_base}.csv"), index=False)